# Fine-tuning QLoRA — Clasificador de riesgo IA (EU AI Act)

> **⚠ Este notebook está diseñado para ejecutarse en Google Colab con GPU T4 (16 GB VRAM).**

Pasos:
0. Instalación de dependencias (solo Colab)
1. Configuración del modelo base y QLoRA
2. Carga del dataset
3. Tokenización
4. Fine-tuning con SFTTrainer
5. Evaluación en test
6. Guardar adaptador LoRA
7. Registro en MLflow

## 0. Instalación (solo Colab)

In [ ]:
# Ejecutar solo en Google Colab
import os
if os.environ.get("COLAB_RELEASE_TAG"):
    !pip install -q transformers==4.47.1 peft==0.14.0 bitsandbytes==0.45.0 trl==0.13.0 datasets==3.2.0 accelerate==1.2.1

## 1. Configuración

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ── Modelo base ──────────────────────────────────────────────────────────────
# Opciones recomendadas para T4 16 GB:
#   - "mistralai/Mistral-7B-v0.1"            (mejor rendimiento general)
#   - "meta-llama/Llama-3.2-3B-Instruct"    (más ligero, menor VRAM)
BASE_MODEL = "mistralai/Mistral-7B-v0.1"

# ── Configuración QLoRA (cuantización 4-bit) ─────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ── Configuración LoRA ────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

print(f"Modelo base: {BASE_MODEL}")
print(f"Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU (no recomendado)'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cargar tokenizador y modelo
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 2. Carga del dataset

In [ ]:
from datasets import load_dataset

# Rutas locales (ajustar si se ejecuta en Colab con Drive montado)
PATH_TRAIN = "data/finetune/train.jsonl"
PATH_TEST  = "data/finetune/test.jsonl"

dataset = load_dataset(
    "json",
    data_files={"train": PATH_TRAIN, "test": PATH_TEST},
)

print(dataset)
print()
print("Ejemplo de muestra:")
print(dataset["train"][0]["text"])

## 3. Tokenización

In [ ]:
MAX_LENGTH = 512

def tokenizar(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

dataset_tokenizado = dataset.map(tokenizar, batched=True)

print(f"Train tokenizado: {len(dataset_tokenizado['train'])} muestras")
print(f"Test tokenizado:  {len(dataset_tokenizado['test'])} muestras")

## 4. Fine-tuning con SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "model/qlora_checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=MAX_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=lora_config,
)

print("Iniciando fine-tuning...")
train_result = trainer.train()

train_loss = train_result.training_loss
print(f"\n✓ Fine-tuning completado")
print(f"  Train loss final: {train_loss:.4f}")

## 5. Evaluación en test

In [ ]:
import re
import numpy as np
from sklearn.metrics import classification_report, f1_score

CLASES = ["inaceptable", "alto_riesgo", "riesgo_limitado", "riesgo_minimo"]

def extraer_etiqueta(texto_generado):
    """Extrae la etiqueta de la respuesta generada por el modelo."""
    texto = texto_generado.lower()
    for clase in CLASES:
        if clase in texto:
            return clase
    return "desconocido"

def predecir(texto_entrada):
    """Genera una predicción para una muestra."""
    # El prompt es todo hasta '### Clasificación:\n'
    prompt = texto_entrada.split("### Clasificación:")[0] + "### Clasificación:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
        )
    respuesta = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extraer_etiqueta(respuesta)

print("Evaluando en test...")
y_true = [row["etiqueta"] for row in dataset["test"]]
y_pred = [predecir(row["text"]) for row in dataset["test"]]

print("\n=== Resultados en TEST (QLoRA) ===")
print(classification_report(y_true, y_pred))

finetune_f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
finetune_accuracy = np.mean([p == t for p, t in zip(y_pred, y_true)])

print(f"F1-macro: {finetune_f1_macro:.4f}")
print(f"Accuracy: {finetune_accuracy:.4f}")

## 6. Guardar adaptador LoRA

In [ ]:
import os

ADAPTER_DIR = "model/qlora_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"✓ Adaptador LoRA guardado en: {ADAPTER_DIR}")
print(f"  Archivos: {os.listdir(ADAPTER_DIR)}")

## 7. Registro en MLflow

In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT   = "clasificador_riesgo_ia"

try:
    if MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="finetune_qlora"):
        # Parámetros
        mlflow.log_params({
            "base_model":                  BASE_MODEL,
            "lora_r":                      lora_config.r,
            "lora_alpha":                  lora_config.lora_alpha,
            "lora_dropout":                lora_config.lora_dropout,
            "lora_target_modules":         str(lora_config.target_modules),
            "bnb_4bit_quant_type":         "nf4",
            "bnb_use_double_quant":        True,
            "num_train_epochs":            training_args.num_train_epochs,
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
            "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
            "learning_rate":               training_args.learning_rate,
            "lr_scheduler_type":           training_args.lr_scheduler_type,
            "max_seq_length":              MAX_LENGTH,
        })

        # Métricas
        mlflow.log_metrics({
            "train_loss":         train_loss,
            "finetune_f1_macro":  finetune_f1_macro,
            "finetune_accuracy":  finetune_accuracy,
        })

        # Artefactos
        mlflow.log_artifacts(ADAPTER_DIR, artifact_path="qlora_adapter")

        print(f"✓ Fine-tuning QLoRA registrado en MLflow")
        print(f"  F1-macro: {finetune_f1_macro:.4f}")
        print(f"  Run ID:   {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")